In [ ]:
# ==========================================
# TEST FRAUD API ON 10 FRAUD + 10 NON FRAUD
# FULL COLAB CODE
# ==========================================

import pandas as pd
import requests

# ------------------------------------------
# 1. Read Dataset
# ------------------------------------------
df = pd.read_csv("/content/final_dataset_after_feature_selection_in_xgboost.csv")


In [ ]:
# ------------------------------------------
# 2. API URL
# ------------------------------------------
url = "https://model-deployment-r4d2.onrender.com/predict"

# ------------------------------------------
# 3. Define FEATURES list
# (same list you used in main.py)
# ------------------------------------------
FEATURES = [col for col in df.columns if col not in ["isFraud", "TransactionDT"]]

# ------------------------------------------
# 4. Select 10 Fraud + 10 Non Fraud
# ------------------------------------------
fraud_df = df[df["isFraud"] == 1].sample(10, random_state=5)
nonfraud_df = df[df["isFraud"] == 0].sample(10, random_state=5)

test_df = pd.concat([fraud_df, nonfraud_df]).sample(frac=1, random_state=42).reset_index(drop=True)

# ------------------------------------------
# 5. Test API Row by Row
# ------------------------------------------
results = []

for i in range(len(test_df)):

    row = test_df.loc[i, FEATURES].to_dict()

    payload = {
        "data": row
    }

    r = requests.post(url, json=payload)

    try:
        output = r.json()
    except:
        output = {"error": r.text}

    actual = int(test_df.loc[i, "isFraud"])

    pred = output.get("prediction", "NA")
    prob = output.get("probability", "NA")

    results.append({
        "Row": i+1,
        "Actual": actual,
        "Prediction": pred,
        "Probability": prob
    })

# ------------------------------------------
# 6. Show Results
# ------------------------------------------
result_df = pd.DataFrame(results)

print(result_df)

# ------------------------------------------
# 7. Accuracy
# ------------------------------------------
correct = (result_df["Actual"] == result_df["Prediction"]).sum()
total = len(result_df)

print("\nTotal Tested :", total)
print("Correct      :", correct)
print("Accuracy     :", round(correct/total*100,2), "%")

    Row  Actual  Prediction  Probability
0     1       1           1     0.880718
1     2       0           0     0.003807
2     3       0           0     0.038465
3     4       1           1     0.897778
4     5       1           1     0.989520
5     6       1           1     0.945100
6     7       0           1     0.969203
7     8       1           0     0.691710
8     9       0           0     0.030589
9    10       0           0     0.031441
10   11       0           0     0.036944
11   12       1           1     0.933933
12   13       1           1     0.956206
13   14       0           0     0.004254
14   15       1           1     0.962407
15   16       0           0     0.131735
16   17       1           1     0.994981
17   18       0           0     0.160987
18   19       0           0     0.001724
19   20       1           1     0.989428

Total Tested : 20
Correct      : 18
Accuracy     : 90.0 %
